# 17.6 Multi-task learning

Shared parameters help when tasks agree on what should be represented and hurt when their gradients fight.

A shared trunk reduces variance across related heads, but task weights decide which gradients shape the representation most.

Save a copy to Drive to edit.

In [ ]:

import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
from sklearn.base import clone
from sklearn.datasets import load_digits
from sklearn.datasets import load_iris
from sklearn.datasets import make_blobs
from sklearn.datasets import make_classification
from sklearn.datasets import make_moons
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
np.random.seed(17)

def budget_ladder():
    """Part 17 (learning paradigms): fix a real base dataset, shrink the LABEL budget per rung.

    Returns [(name, X, y, labeled_mask), ...] over the SAME digits data, only the fraction of
    labeled points falls D1->D5. The 'watch it scale' curve is accuracy vs label budget.
    """
    digits = load_digits()
    X = digits.data / 16.0
    y = digits.target
    rng = np.random.default_rng(17)
    rungs = []
    for name, frac in [("D1 100% labels", 1.0), ("D2 50% labels", 0.5), ("D3 20% labels", 0.2), ("D4 5% labels", 0.05), ("D5 2% labels", 0.02)]:
        mask = rng.random(y.shape) < frac
        if mask.sum() < 20:
            idx = rng.choice(len(y), size=20, replace=False)
            mask = np.zeros(len(y), dtype=bool)
            mask[idx] = True
        rungs.append((name, X, y, mask))
    return rungs


def split_labeled_train_test(X, y, labeled_mask, seed=0):
    train_idx, test_idx = train_test_split(
        np.arange(len(y)),
        test_size=0.35,
        random_state=seed,
        stratify=y,
    )
    labeled_idx = train_idx[labeled_mask[train_idx]]
    if len(np.unique(y[labeled_idx])) < len(np.unique(y)):
        rng = np.random.default_rng(seed)
        needed = []
        for cls in np.unique(y):
            choices = train_idx[y[train_idx] == cls]
            needed.append(rng.choice(choices))
        labeled_idx = np.unique(np.concatenate([labeled_idx, np.array(needed)]))
    return labeled_idx, train_idx, test_idx

def fit_logistic_accuracy(X, y, labeled_mask, seed=0):
    labeled_idx, train_idx, test_idx = split_labeled_train_test(X, y, labeled_mask, seed=seed)
    scaler = StandardScaler()
    scaler.fit(X[train_idx])
    X_labeled = scaler.transform(X[labeled_idx])
    X_test = scaler.transform(X[test_idx])
    clf = LogisticRegression(max_iter=1500, C=2.0, solver="lbfgs")
    clf.fit(X_labeled, y[labeled_idx])
    pred = clf.predict(X_test)
    return accuracy_score(y[test_idx], pred), clf, scaler, labeled_idx, test_idx

def ensure_class_budget_mask(y, frac, seed):
    rng = np.random.default_rng(seed)
    mask = np.zeros(len(y), dtype=bool)
    for cls in np.unique(y):
        cls_idx = np.where(y == cls)[0]
        count = max(1, int(round(frac * len(cls_idx))))
        chosen = rng.choice(cls_idx, size=count, replace=False)
        mask[chosen] = True
    return mask

def two_dimensional_view(X, seed=0):
    if X.shape[1] == 2:
        return X
    return PCA(n_components=2, random_state=seed).fit_transform(StandardScaler().fit_transform(X))

def plot_panel(ax, X, y, title, marked=None):
    Z = two_dimensional_view(X)
    ax.scatter(Z[:, 0], Z[:, 1], c=y, s=12, cmap="tab10", alpha=0.65)
    if marked is not None and len(marked) > 0:
        M = Z[marked]
        ax.scatter(M[:, 0], M[:, 1], facecolors="none", edgecolors="black", s=45, linewidths=1.0)
    ax.set_title(title, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])


## The concept, built once

The objective $L=\sum_tlpha_tL_t$ is numeric, not decorative. The lesson's weights produce $0.028+0.027=0.055$.

In [ ]:

def weighted_multitask_loss(losses, alpha):
    losses = np.asarray(losses, dtype=float)
    weights = np.array([alpha, 1.0 - alpha])
    parts = weights * losses
    return parts, float(parts.sum())

parts, total = weighted_multitask_loss([0.04, 0.09], alpha=0.7)

assert np.allclose(np.round(parts, 3), [0.028, 0.027])
assert round(total, 3) == 0.055
print("weighted parts=", np.round(parts, 3))
print(f"total={total:.3f}")


The real algorithm is a small shared logistic trunk implemented in NumPy. It trains one sigmoid head per task and reports the primary parity-task accuracy.

In [ ]:

def sigmoid(a):
    return 1.0 / (1.0 + np.exp(-a))

def train_shared_multitask(X, tasks, weights, seed=0, steps=180, hidden=12, lr=0.25):
    rng = np.random.default_rng(seed)
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    n, d = Xs.shape
    task_count = len(tasks)
    W = rng.normal(0.0, 0.12, size=(d, hidden))
    heads = rng.normal(0.0, 0.12, size=(task_count, hidden))
    for step in range(steps):
        H = np.tanh(Xs @ W)
        grad_W = np.zeros_like(W)
        grad_heads = np.zeros_like(heads)
        for t, target in enumerate(tasks):
            prob = sigmoid(H @ heads[t])
            error = (prob - target) * weights[t] / n
            grad_heads[t] += H.T @ error
            hidden_error = (error[:, None] * heads[t][None, :]) * (1.0 - H ** 2)
            grad_W += Xs.T @ hidden_error
        W -= lr * grad_W
        heads -= lr * grad_heads
    H = np.tanh(Xs @ W)
    primary_prob = sigmoid(H @ heads[0])
    pred = (primary_prob >= 0.5).astype(int)
    return pred, W, heads, scaler

def multitask_score(X, y_digit, task_specs, seed=0):
    y_primary = (y_digit % 2).astype(int)
    train_idx, test_idx = train_test_split(np.arange(len(y_digit)), test_size=0.35, random_state=seed, stratify=y_primary)
    tasks_train = []
    tasks_all = []
    weights = []
    for name, values, weight in task_specs:
        tasks_train.append(values[train_idx])
        tasks_all.append(values)
        weights.append(weight)
    total = sum(weights)
    weights = [w / total for w in weights]
    pred_train, W, heads, scaler = train_shared_multitask(X[train_idx], tasks_train, weights, seed=seed)
    X_test = scaler.transform(X[test_idx])
    H_test = np.tanh(X_test @ W)
    prob = sigmoid(H_test @ heads[0])
    pred = (prob >= 0.5).astype(int)
    acc = accuracy_score(y_primary[test_idx], pred)
    return acc, W, heads, train_idx, test_idx

def make_multitask_ladder():
    digits = load_digits()
    X = digits.data / 16.0
    yd = digits.target
    parity = (yd % 2).astype(int)
    high = (yd >= 5).astype(int)
    loopish = np.isin(yd, [0, 6, 8, 9]).astype(int)
    nuisance = 1 - parity
    specs = [
        ("D1 primary only", [("parity", parity, 1.0)]),
        ("D2 related high/low", [("parity", parity, 0.7), ("high", high, 0.3)]),
        ("D3 add loop task", [("parity", parity, 0.6), ("high", high, 0.25), ("loop", loopish, 0.15)]),
        ("D4 four related heads", [("parity", parity, 0.55), ("high", high, 0.2), ("loop", loopish, 0.15), ("round", loopish, 0.10)]),
        ("D5 adversarial nuisance", [("parity", parity, 0.35), ("nuisance", nuisance, 0.65)]),
    ]
    return [(name, X, yd, task_specs) for name, task_specs in specs]


## The dataset ladder

The ladder adapts the modifier to number and conflict of tasks on real digits. D5 intentionally overweights an adversarial nuisance head.

In [ ]:

mtl_rungs = make_multitask_ladder()

for name, X, yd, task_specs in mtl_rungs:
    print(f"{name:26s} X={X.shape} tasks={len(task_specs)} weights={[round(t[2], 2) for t in task_specs]}")


In [ ]:

mtl_results = []

for name, X, yd, task_specs in mtl_rungs:
    acc, W, heads, train_idx, test_idx = multitask_score(X, yd, task_specs, seed=8)
    conflict = len(task_specs) - 1
    if "adversarial" in name:
        conflict = 4
    mtl_results.append((name, conflict, acc, X, yd, train_idx))
    print(f"{name:26s} primary_accuracy={acc:.3f}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(14, 5))

for ax, row in zip(axes[0], mtl_results):
    name, conflict, acc, X, yd, train_idx = row
    plot_panel(ax, X, yd % 2, name.split()[0] + f" acc={acc:.2f}", marked=train_idx[:40])

conflicts = [row[1] for row in mtl_results]
accuracies = [row[2] for row in mtl_results]
axes[1, 0].plot(conflicts, accuracies, marker="o")
axes[1, 0].set_xlabel("task conflict index")
axes[1, 0].set_ylabel("primary accuracy")
axes[1, 0].set_title("accuracy vs tasks/conflict")

for ax in axes[1, 1:]:
    ax.axis("off")

plt.tight_layout()
plt.show()


## Pitfall on D5: gradient fight

The adversarial nuisance head asks the shared trunk to encode the opposite of the primary task. Reducing its weight or removing it fixes negative transfer.

In [ ]:

name, X, yd, task_specs = mtl_rungs[-1]
bad_acc, _, _, _, _ = multitask_score(X, yd, task_specs, seed=8)
parity = (yd % 2).astype(int)
nuisance = 1 - parity
fixed_specs = [("parity", parity, 0.85), ("nuisance", nuisance, 0.15)]
fixed_acc, _, _, _, _ = multitask_score(X, yd, fixed_specs, seed=8)
removed_specs = [("parity", parity, 1.0)]
removed_acc, _, _, _, _ = multitask_score(X, yd, removed_specs, seed=8)

print(f"D5 overweight nuisance accuracy={bad_acc:.3f}")
print(f"D5 downweighted nuisance accuracy={fixed_acc:.3f}")
print(f"D5 removed nuisance accuracy={removed_acc:.3f}")
assert max(fixed_acc, removed_acc) >= bad_acc - 0.05


## Evaluate it + Practice

- Metric: primary-task accuracy vs number/conflict of tasks, always compared with a no-skill majority or scratch baseline.
- Sanity check: labels must be shuffled or withheld to confirm the method loses its advantage.
- Ablation: turn off the key paradigm idea and verify the metric drops.
- Failure signal: the diagnostic score improves while held-out target accuracy falls.
- Robustness check: repeat with a different seed and inspect the hardest rung.

### Practice 1
Change the budget or shift on D4 and rerun the summary curve.

### Practice 2
Add one ablation that removes the paradigm-specific step.

### Practice 3
Write a one-sentence deployment warning for D5.